In [ ]:
!pip -q install --force-reinstall --no-deps \
    'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' \
    --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3'

In [ ]:
import os
import subprocess
import sys

REPO = "/kaggle/working/lawforge"
if not os.path.isdir(REPO):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/PAMF2/lawforge.git", REPO]
    )
sys.path.insert(0, REPO)
print(
    "repo HEAD:",
    subprocess.check_output(["git", "-C", REPO, "log", "-1", "--oneline"])
    .decode()
    .strip(),
)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE = "Goedel-LM/Goedel-Prover-V2-8B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="cuda", trust_remote_code=True
)
model.eval()
print(f"loaded {BASE} mem={torch.cuda.memory_allocated() / 1e9:.2f}GB")

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f"{REPO}/kaggle/llm_classify/inputs")
rows = []
for s in ["hard2_test", "hard3_test"]:
    for line in open(INPUTS / f"{s}.jsonl"):
        r = json.loads(line)
        r["_split"] = s
        rows.append(r)
print(f"rows: {len(rows)}")

In [ ]:
import time
import json
from pathlib import Path

PROMPT = (
    "You are a magma theorist. A magma is a set with one binary operation \u25c7.\n"
    "Given hypothesis h (universally true for all variables) and goal g, decide:\n"
    "does h imply g for ALL magmas? Answer ONLY with the single word TRUE or FALSE.\n\n"
    "Hypothesis: {eq1}\n"
    "Goal: {eq2}\n\n"
    "Answer:"
)


def to_diamond(s):
    return s.replace("*", "\u25c7")


@torch.inference_mode()
def classify(eq1, eq2):
    p = PROMPT.format(eq1=to_diamond(eq1), eq2=to_diamond(eq2))
    inputs = tok(p, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    txt = (
        tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        .strip()
        .upper()
    )
    if "FALSE" in txt:
        return "false", txt
    if "TRUE" in txt:
        return "true", txt
    return "unknown", txt


OUT = Path("/kaggle/working/llm_preds.jsonl")
t0 = time.time()
stats = {"tp_t": 0, "fp_t": 0, "tp_f": 0, "fp_f": 0, "unk": 0}
with OUT.open("w") as f:
    for i, r in enumerate(rows):
        pred, raw = classify(r["hypothesis"], r["goal"])
        label = r["label"]
        if pred == "true":
            stats["tp_t" if label == "true" else "fp_t"] += 1
        elif pred == "false":
            stats["tp_f" if label == "false" else "fp_f"] += 1
        else:
            stats["unk"] += 1
        f.write(
            json.dumps(
                {
                    "id": r["id"],
                    "split": r["_split"],
                    "label": label,
                    "pred": pred,
                    "raw": raw,
                }
            )
            + "\n"
        )
        f.flush()
        if (i + 1) % 50 == 0:
            correct = stats["tp_t"] + stats["tp_f"]
            print(
                f"[{i + 1}/{len(rows)}] correct={correct} stats={stats} t={time.time() - t0:.0f}s",
                flush=True,
            )

correct = stats["tp_t"] + stats["tp_f"]
print("=== LLM CLASSIFY FINAL ===", flush=True)
print(f"stats: {stats}", flush=True)
print(
    f"accuracy (true+false correct only): {correct}/{len(rows)} = {correct / len(rows) * 100:.1f}%",
    flush=True,
)
print(f"total time: {time.time() - t0:.0f}s", flush=True)